# Chapter 07: Advanced Text Generation Techniques and tools

In [2]:
%%capture
!pip install langchain openai langchain_openai transformers datasets accelerate sentence-transformers duckduckgo-search langchain_community

# Fix: Use GGML_CUDA=on instead of LLAMA_CUDA=on
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.2.69 --force-reinstall --no-cache-dir


# Loading the model

In [3]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-08-25 04:24:00--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 52.85.193.32, 52.85.193.24, 52.85.193.123, ...
Connecting to huggingface.co (huggingface.co)|52.85.193.32|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1787635440&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJs

In [4]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp

print("Downloading model from Hugging Face Hub (approx. 7.6GB)...")
# Safely download the precise quantized model file
model_local_path = hf_hub_download(
    repo_id="Microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="Phi-3-mini-4k-instruct-fp16.gguf"
)
print(f"Download complete! File saved to: {model_local_path}")

print("Loading model into GPU VRAM...")
# Pass the verified download path directly into LlamaCpp
llm = LlamaCpp(
    model_path=model_local_path,
    n_gpu_layers=-1,   # Offloads all layers to your Colab GPU
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

print("🚀 Success! Your Phi-3 model is fully loaded and ready to use.")


Phi-3-mini-4k-instruct-fp16.gguf:   0%|          | 0.00/7.64G [00:00<?, ?B/s]

Download complete! File saved to: /root/.cache/huggingface/hub/models--Microsoft--Phi-3-mini-4k-instruct-gguf/snapshots/a64113399c2f6b8ad3e11c394733a2ddadaa7f33/Phi-3-mini-4k-instruct-fp16.gguf
Loading model into GPU VRAM...
🚀 Success! Your Phi-3 model is fully loaded and ready to use.


In [5]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

## Chains

In [6]:
from langchain_core.prompts import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [7]:
basic_chain = prompt | llm

In [8]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

' Hello Maarten, the answer to 1 + 1 is 2.'

## Multiple Chains

In [12]:
%%capture
!pip install langchain-classic

In [15]:
from langchain_classic.chains import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

/tmp/ipykernel_58/1360148921.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [16]:
title.invoke({"summary": "a girl that lost her mother"})

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of Love: A Journey Through Grief"'}

In [17]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [18]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [19]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [20]:
llm_chain.invoke("a girl that lost her mother")

{'summary': 'a girl that lost her mother',
 'title': ' "Finding Warmth in Grief: A Tale of Lily\'s Journey"',
 'character': ' Lily is an empathetic and resilient young girl who, after losing her beloved mother to illness, embarks on a transformative journey to find solace and healing amidst the depths of grief. Her kind heart and unyielding spirit lead her through trials, allowing her to discover newfound strength in cherishing memories while also seeking comfort within herself.',
 'story': " Finding Warmth in Grief: A Tale of Lily's Journey tells the heartfelt tale of an empathetic and resilient young girl, named Lily. After losing her beloved mother to illness, she embarks on a transformative journey seeking solace and healing amidst the depths of grief. Along the way, Lily's kind heart guides her through trials that allow her to discover newfound strength in cherishing memories while also finding comfort within herself. Through acts of compassion towards others who share similar los

# Memory

In [21]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

" The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units altogether."

In [22]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

" I'm unable to determine your name as I don't have the capability to access personal data. If you need assistance with something else, feel free to ask!"

## ConversationBuffer

In [23]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [25]:
from langchain_classic.memory import ConversationBufferMemory

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_58/3768838688.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history")


In [27]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': 'Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  Hello Maarten, the answer to your math question is 2. 1 plus 1 equals 2. How can I assist you further today?',
 'text': " Greetings Maarten, a simple addition problem gives us an answer of 2 for 1 + 1. Can I help with any other questions or topics you're curious about?\n\nNow that we know the solution to your math question:\n\n1 + 1 = 2\n\nFeel free to ask anything else!"}

In [28]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  Hello Maarten, the answer to your math question is 2. 1 plus 1 equals 2. How can I assist you further today?\nHuman: Hi! My name is Maarten. What is 1 + 1?\nAI:  Greetings Maarten, a simple addition problem gives us an answer of 2 for 1 + 1. Can I help with any other questions or topics you're curious about?\n\nNow that we know the solution to your math question:\n\n1 + 1 = 2\n\nFeel free to ask anything else!",
 'text': ' Your name is Maarten, as mentioned in our current conversation. How may I assist you further?\n{}'}

## ConversationBufferMemoryWindow

In [29]:
from langchain_classic.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_58/1769901941.py:4: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")


In [30]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten, my name isn't programmed to remember personal details like age for privacy reasons, but I can certainly help you with the math question. The answer to 1 + 1 is 2.",
 'text': ' The result of 3 + 3 is 6.'}

In [31]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten, my name isn't programmed to remember personal details like age for privacy reasons, but I can certainly help you with the math question. The answer to 1 + 1 is 2.\nHuman: What is 3 + 3?\nAI:  The result of 3 + 3 is 6.",
 'text': " Your name, as mentioned in the conversation, is Maarten.\n\nAs for the math question, your answer to 3 + 3 would be 6. However, it's important to note that I don't actually know any personal details about you, including your name or age, as I prioritize user privacy and data protection."}

In [33]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': "Human: What is my name?\nAI:  Your name, as mentioned in the conversation, is Maarten.\n\nAs for the math question, your answer to 3 + 3 would be 6. However, it's important to note that I don't actually know any personal details about you, including your name or age, as I prioritize user privacy and data protection.\nHuman: What is my age?\nAI:  I'm unable to determine your age as I don't have access to such personal information. Age is private, and it's important for maintaining individual privacy. If you need assistance with any other general knowledge or non-personal queries, feel free to ask!",
 'text': " I apologize for the confusion earlier. As an AI, I don't have access to personal information such as your age. If you have any general questions or need assistance, I'd be happy to help! However, please remember that sharing sensitive personal details online should be done cautiously and within privacy guidelines."}

## ConversationSummary

In [39]:
# Create a summary prompt template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [40]:
from langchain_classic.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [41]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' Hello Maarten, the sum of 1 + 1 equals 2. How may I provide further assistance to you today?',
 'text': ' Your name appears to be not directly mentioned in this conversation. However, based on your statement "Hello," it suggests that you might have addressed me as Maarten or a similar formality. As the assistant, I don\'t have personal names but you can simply refer to yourself as the user. How may I assist you further?'}

In [42]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': " Hello, as the conversation does not directly mention your name, it seems like you referred to me as Maarten or a similar formality. You can call me an AI assistant and ask for any assistance you need since I don't have personal names. The established fact that 1 + 1 equals 2 was also reaffirmed. How may I further assist you?",
 'text': ' The first question you asked was, "Hello, as the conversation does not directly mention your name, it seems like you referred to me as Maarten or a similar formality. You can call me an AI assistant and ask for any assistance you need since I don\'t have personal names."'}

In [44]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': ' The initial conversation involved the human addressing the AI with formalities such as "Maarten" instead of their name, identifying themselves as an AI assistant. They confirmed that 1 + 1 equals 2 and requested a recap of their first question. The AI summarized the conversation by stating that the user\'s initial inquiry was about referring to them formally without using personal names.'}

# Agents